<a href="https://colab.research.google.com/github/SakshiKhatiwada/Python/blob/main/Deep-Learning/keras_hyperparameter_tuning_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("diabetes.csv")

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [5]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [6]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [7]:
X = scaler.fit_transform(X)

In [8]:
X.shape

(768, 8)

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=52 )

In [10]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense

In [11]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=8))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='Adam', loss='binary_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
model.fit(X_train, y_train, batch_size= 32, epochs = 100, validation_data=(X_test, y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5349 - loss: 0.6957 - val_accuracy: 0.6429 - val_loss: 0.6364
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6916 - loss: 0.6480 - val_accuracy: 0.7338 - val_loss: 0.5919
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7447 - loss: 0.5699 - val_accuracy: 0.7403 - val_loss: 0.5596
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7635 - loss: 0.5554 - val_accuracy: 0.7597 - val_loss: 0.5376
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7456 - loss: 0.5357 - val_accuracy: 0.7532 - val_loss: 0.5219
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7805 - loss: 0.5123 - val_accuracy: 0.7468 - val_loss: 0.5102
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7718 - loss: 0.5093 - val_accuracy: 0.7532 - val_loss: 0.5016
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7849 - loss: 0.4909 - val_accuracy: 0.7662 - 

In [13]:
!pip install -U keras-tuner
import kerastuner as kt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 8.7 MB/s eta 0:00:00


/tmp/ipython-input-3209566528.py:2: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


In [14]:
def build_model(hp):
  model = Sequential()
  model.add(Dense(32, activation='relu', input_dim=8))
  model.add(Dense(1, activation='sigmoid'))

  optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop', 'adadelta'])
  model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [15]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5
    )

In [16]:
tuner.search(X_train,y_train, epochs=5, validation_data=(X_test, y_test))

Trial 4 Complete [00h 00m 07s]
val_accuracy: 0.4675324559211731

Best val_accuracy So Far: 0.7727272510528564
Total elapsed time: 00h 00m 23s


In [17]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [18]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [19]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
model.fit(X_train, y_train, batch_size=32, epochs=100, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.7476 - loss: 0.5622 - val_accuracy: 0.7792 - val_loss: 0.5264
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7605 - loss: 0.5346 - val_accuracy: 0.7792 - val_loss: 0.5118
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7796 - loss: 0.4971 - val_accuracy: 0.7727 - val_loss: 0.5017
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7960 - loss: 0.4849 - val_accuracy: 0.7662 - val_loss: 0.4958
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7738 - loss: 0.4944 - val_accuracy: 0.7662 - val_loss: 0.4903
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7870 - loss: 0.4667 - val_accuracy: 0.7792 - val_loss: 0.4875
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7902 - loss: 0.4632 - val_accuracy: 0.7662 - val_loss: 0.4868
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7659 - loss: 0.4940 - val_accuracy: 0.7

In [21]:
# to find the optimal no. of nodes

def build_model2(hp):

  model = Sequential()
  units = hp.Int('units', min_value=8, max_value=128, step=8) # lower and upper limit, step size
  model.add(Dense(units = units, activation='relu', input_dim = 8))
  model.add(Dense(1, activation='sigmoid'))

  model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [22]:
tuner = kt.RandomSearch(build_model2, objective='val_accuracy', max_trials=5,
                        directory='mydir')

In [23]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Trial 5 Complete [00h 00m 02s]
val_accuracy: 0.7922077775001526

Best val_accuracy So Far: 0.7922077775001526
Total elapsed time: 00h 00m 12s


In [24]:
tuner.get_best_hyperparameters()[0].values

{'units': 56}

In [25]:
model = tuner.get_best_models(num_models=1)[0]

In [26]:
model.fit(X_train, y_train, batch_size=32, epochs = 100, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7544 - loss: 0.4940 - val_accuracy: 0.7987 - val_loss: 0.4860
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7451 - loss: 0.5019 - val_accuracy: 0.8052 - val_loss: 0.4834
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7661 - loss: 0.4581 - val_accuracy: 0.7987 - val_loss: 0.4829
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7577 - loss: 0.4860 - val_accuracy: 0.7987 - val_loss: 0.4823
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7728 - loss: 0.4675 - val_accuracy: 0.7922 - val_loss: 0.4815
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7745 - loss: 0.4705 - val_accuracy: 0.7922 - val_loss: 0.4820
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7745 - loss: 0.4635 - val_accuracy: 0.7922 - val_loss: 0.4813
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8005 - loss: 0.4401 - val_accuracy: 0.77

In [27]:
# To Select the no. of layers
def build_model3(hp):
  model = Sequential()
  model.add(Dense(72, activation='relu', input_dim=8))

  for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
    model.add(Dense(72, activation='relu'))

  model.add(Dense(1, activation='sigmoid'))
  model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
  return model


In [28]:
tuner = kt.RandomSearch(build_model3,
                        objective='val_accuracy',
                        max_trials=3,
                        directory='mydir',
                        project_name='hero-me')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [29]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.7792207598686218

Best val_accuracy So Far: 0.7857142686843872
Total elapsed time: 00h 00m 10s


In [31]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 8 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [32]:
model.fit(X_train, y_train, batch_size=32, epochs=100, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7488 - loss: 0.5136 - val_accuracy: 0.7857 - val_loss: 0.4773
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7744 - loss: 0.4524 - val_accuracy: 0.7922 - val_loss: 0.4805
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7780 - loss: 0.4429 - val_accuracy: 0.7727 - val_loss: 0.4834
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7704 - loss: 0.4737 - val_accuracy: 0.7857 - val_loss: 0.4836
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7859 - loss: 0.4354 - val_accuracy: 0.7857 - val_loss: 0.4853
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7635 - loss: 0.4627 - val_accuracy: 0.7792 - val_loss: 0.4924
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7893 - loss: 0.4348 - val_accuracy: 0.7792 - val_loss: 0.4926
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8097 - loss: 0.4062 - val_accuracy: 0.77

In [62]:
from keras.layers import Dropout

In [63]:
# all in one model

def build_model_all(hp):
  model = Sequential()
  counter = 0

  for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
    if counter == 0:
      model.add(Dense(hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
                      activation=hp.Choice('activation' + str(i), values=['relu', 'tanh', 'sigmoid']),
                      input_dim=8
                      ))
      model.add(Dropout(hp.Choice('dropout'+str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])))

    else:
      model.add(Dense(hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
                      activation=hp.Choice('activation' + str(i), values=['relu', 'tanh', 'sigmoid']),
      ))
      model.add(Dropout(hp.Choice('dropout'+str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])))

    counter += 1

  model.add(Dense(1, activation='sigmoid'))
  model.compile(optimizer = hp.Choice('optimizer', values=['rmsprop', 'adam', 'sgd', 'adadelta', 'nadam']),
                loss='binary_crossentropy',
                metrics=['accuracy'])

  return model

In [64]:
tuner = kt.RandomSearch(build_model_all,
                        objective='val_accuracy',
                        max_trials=3,
                        directory='mydir',
                        project_name='num_layers_dropout')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [65]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.7337662577629089

Best val_accuracy So Far: 0.7337662577629089
Total elapsed time: 00h 00m 19s


In [66]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 1,
 'units0': 88,
 'activation0': 'relu',
 'dropout0': 0.9,
 'optimizer': 'sgd',
 'units1': 24,
 'activation1': 'relu',
 'dropout1': 0.1,
 'units2': 112,
 'activation2': 'sigmoid',
 'dropout2': 0.9,
 'units3': 24,
 'activation3': 'tanh',
 'dropout3': 0.8,
 'units4': 48,
 'activation4': 'relu',
 'dropout4': 0.6,
 'units5': 56,
 'activation5': 'sigmoid',
 'dropout5': 0.4,
 'units6': 16,
 'activation6': 'tanh',
 'dropout6': 0.2,
 'units7': 48,
 'activation7': 'relu',
 'dropout7': 0.5,
 'units8': 16,
 'activation8': 'relu',
 'dropout8': 0.5}

In [67]:
model = tuner.get_best_models(num_models=1)[0]

In [68]:
model.fit(X_train, y_train, epochs=200, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5721 - loss: 0.7449 - val_accuracy: 0.7338 - val_loss: 0.5879
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5611 - loss: 0.7150 - val_accuracy: 0.7338 - val_loss: 0.5785
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6505 - loss: 0.6313 - val_accuracy: 0.7403 - val_loss: 0.5701
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6464 - loss: 0.6593 - val_accuracy: 0.7532 - val_loss: 0.5623
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6453 - loss: 0.6530 - val_accuracy: 0.7532 - val_loss: 0.5560
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6884 - loss: 0.6099 - val_accuracy: 0.7597 - val_loss: 0.5518
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5984 - loss: 0.6792 - val_accuracy: 0.7727 - val_loss: 0.5464
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6712 - loss: 0.6179 - val_accuracy: 0.76